In [1]:
import pandas as pd
from schema import enforce_schema, coerce_types

In [29]:
df = []

In [30]:
# Import the raw data:
df = pd.read_csv("../../data/russell2024.csv")
df.head()

,id,infection_id,swab_date,swab_type,ct_unadjusted,result,VOC,symptoms,symptom_onset_date,total_infections,...,age_group,ct_type,ct_value,t_since_last_dose,t_first_inf_by_id,t_first_inf_to_cur_inf,t_since_last_exposure,t_first_test,data_id,swab_type_num
0,1,1,2/10/22,Dry,NaN,Negative,Omicron (BA.2),symptomatic,2/18/22,1,...,20-34,ct_value,40.000000,46,2/17/22,0,0.147059,-7,1,0
1,1,1,2/17/22,Dry,25.031553,Positive,Omicron (BA.2),symptomatic,2/18/22,1,...,20-34,ct_value,25.031553,46,2/17/22,0,0.147059,0,1,0
2,1,1,2/22/22,Dry,18.954472,Positive,Omicron (BA.2),symptomatic,2/18/22,1,...,20-34,ct_value,18.954472,46,2/17/22,0,0.147059,5,1,0
3,1,1,2/24/22,Dry,17.020020,Positive,Omicron (BA.2),symptomatic,2/18/22,1,...,20-34,ct_value,17.020020,46,2/17/22,0,0.147059,7,1,0
4,1,1,3/11/22,Dry,29.461030,Positive,Omicron (BA.2),symptomatic,2/18/22,1,...,20-34,ct_value,29.461030,46,2/17/22,0,0.147059,22,1,0


In [31]:
# Keep only the columns we need: 
df = df[['id', 'swab_type', 'VOC', 'symptoms', 'symptom_onset_date', 't', 'age_group', 'ct_type', 'ct_value']]

# Format the age group column into separate age ranges: 
df["AgeRng1"] = df["age_group"].map({
    "20-34": 20,
    "35-49": 35,
    "50+": 50
    })
df["AgeRng2"] = df["age_group"].map({
    "20-34": 34,
    "35-49": 49,
    "50+": 100
    })

# Convert to numeric
df["AgeRng1"] = pd.to_numeric(df["AgeRng1"], errors="coerce")
df["AgeRng2"] = pd.to_numeric(df["AgeRng2"], errors="coerce")

df.head()

,id,swab_type,VOC,symptoms,symptom_onset_date,t,age_group,ct_type,ct_value,AgeRng1,AgeRng2
0,1,Dry,Omicron (BA.2),symptomatic,2/18/22,-7,20-34,ct_value,40.000000,20,34
1,1,Dry,Omicron (BA.2),symptomatic,2/18/22,0,20-34,ct_value,25.031553,20,34
2,1,Dry,Omicron (BA.2),symptomatic,2/18/22,5,20-34,ct_value,18.954472,20,34
3,1,Dry,Omicron (BA.2),symptomatic,2/18/22,7,20-34,ct_value,17.020020,20,34
4,1,Dry,Omicron (BA.2),symptomatic,2/18/22,22,20-34,ct_value,29.461030,20,34


In [36]:
# check what possible values in df["swab_type"] are
df["swab_type"].value_counts()

swab_type
Dry    1154
VTM     297
Name: count, dtype: int64

In [ ]:
df["Targets"] = df["ct_type"].map({
    "ct_value": "ORF1a",
    "ct_n_gene": "N",
    "ct_s_gene": "S"
    })

# Rename columns to match schema: 
df = df.rename(columns={
    "id": "PersonID",
    "swab_type": "SampleMethod",
    "VOC": "Subtype",
    "symptoms": "Symptoms1",
    "t": "TimeDays",
    "ct_value": "Log10VL"
    })

df["SampleMethod"] = df["SampleMethod"].map({
    "Dry": "dry_swab",
    "VTM": "swab_in_VTM",
    })

# df = split_age_range(df, col="age_group")

# Add additional columns with known but missing information:
df["StudyID"] = "russell2024"
df["Pathogen"] = "SARS2"
df["PtSpecies"] = "Human"
df["DOI"] = "10.1371/journal.pbio.3002463"
df["Units"] = "Ct"
df["SampleSource"] = "nasopharyngeal"
df["PlatformName"] = "RT-qPCR"
df["PlatformTech"] = "QuantStudio 3"

In [ ]:
df = enforce_schema(df)
df = coerce_types(df)

df.head()